*cell-1*

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok transformers peft accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00


*cell-2*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


*cell-3*

In [4]:
import threading
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# إعداد المسارات
DRIVE_MODEL_PATH = "/content/drive/MyDrive/HealthLens-Pro-Tuned"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

print("Loading Base Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Loading Your Trained Model from Drive...")
model = PeftModel.from_pretrained(base_model, DRIVE_MODEL_PATH)
model.eval()

app = FastAPI()

class ChatRequest(BaseModel):
    question: str

@app.post("/chat")
def chat_with_healthlens(request: ChatRequest):
    prompt = f"### Instruction:\nYou are HealthLens, a medical assistant. Answer the user's question safely.\n\n### User:\n{request.question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    final_answer = response.split("### Assistant:")[-1].strip() if "### Assistant:" in response else response
    return {"reply": final_answer}

# إعداد Ngrok (ضع التوكن الخاص بك هنا)
NGROK_AUTH_TOKEN = "3BDgRFFN2XtiC68l4iCFFgYgpuF_ApFTgYZpXFkBj77M2rz6"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url
print(f"\n✅ Copy this URL to your React Native app:")
print(f"🔗 {public_url}/chat\n")

# حل مشكلة الـ RuntimeError باستخدام threading
def run_api():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

Loading Base Model and Tokenizer...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Your Trained Model from Drive...

✅ Copy this URL to your React Native app:
🔗 https://karan-pseudooriental-callum.ngrok-free.dev/chat

